# 02 · EDA step 1: choose k for `TimeSeriesSplit` (women's)

**Pre-registered rule** (written before running, not changed after; see `docs/eda_plan_women.md`):
1. **Eligible k:** every test fold and the first training set have at least 100 games (the men's 150, scaled to ~600 games).
2. For each eligible k, rank four **fixed** check features by their mean log-loss gain over
   the baseline, across folds. The features are season-to-date goal differential, shot
   share, PP% and save %.
3. Choose the **smallest eligible k whose ranking agrees with every larger eligible k**
   (Spearman ≥ 0.8). **Tie-break** (the men's amended rule, applied in advance here): among
   k values whose rankings agree exactly, take the largest, to leave the most folds for step 5.
4. If none qualifies, use **k = 4**.

This notebook prints **stability numbers only**. It never shows feature names next to their
scores, so this step can't steer which features get favored later.

In [1]:
%run ./01_pregame_features.ipynb

/var/folders/72/ygtskn4x6gq5fmq9wy6cvllw0000gn/T/ipykernel_44205/2041251285.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["home_win"] = out["result"].map({"W": 1, "L": 0})  # NaN for ties
/var/folders/72/ygtskn4x6gq5fmq9wy6cvllw0000gn/T/ipykernel_44205/2041251285.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return out.reset_index().sort_values(["game_date", "game_id"]).reset_index(drop=True)
/var/folders/72/ygtskn4x6gq5fmq9wy6cvllw0000gn/T/ipykernel_44205/2041251285.py:39: PerformanceWarning: DataFrame is high

All leakage checks passed.


2025-26: 696 games, 41 ties
After warm-up (both teams >= 3 GP) and dropping ties: 579 games
59 diff_ features; missing values after warm-up: 0


/var/folders/72/ygtskn4x6gq5fmq9wy6cvllw0000gn/T/ipykernel_44205/2041251285.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["home_win"] = out["result"].map({"W": 1, "L": 0})  # NaN for ties
/var/folders/72/ygtskn4x6gq5fmq9wy6cvllw0000gn/T/ipykernel_44205/2041251285.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return out.reset_index().sort_values(["game_date", "game_id"]).reset_index(drop=True)


In [2]:
import warnings

import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CHECK_FEATURES = [
    "diff_goal_diff_pg_season",
    "diff_shot_share_season",
    "diff_pp_pct_season",
    "diff_save_pct_season",
]
K_RANGE = range(3, 9)
MIN_FOLD = 100  # scaled: about a sixth of ~600 games (docs/eda_plan_women.md)
AGREEMENT = 0.8
DEFAULT_K = 4  # scaled default

## Helpers

In [3]:
def fold_sizes(n: int, k: int) -> tuple[int, int]:
    """(first training set, smallest test fold) for TimeSeriesSplit(k) on n rows."""
    splits = list(TimeSeriesSplit(n_splits=k).split(np.zeros(n)))
    return len(splits[0][0]), min(len(test) for _, test in splits)


def eligible(n: int, ks=K_RANGE, min_fold: int = MIN_FOLD) -> list[int]:
    return [k for k in ks if min(fold_sizes(n, k)) >= min_fold]


def fold_scores(df: pd.DataFrame, k: int, features: list[str]) -> pd.DataFrame:
    """Log-loss gain over the baseline per fold (rows) and feature (columns).

    The baseline always predicts the training win rate. Positive = the feature beat it.
    """
    y = df["home_win"].to_numpy()
    rows = []
    for train, test in TimeSeriesSplit(n_splits=k).split(df):
        base = log_loss(y[test], np.full(len(test), y[train].mean()), labels=[0, 1])
        row = {}
        for f in features:
            x = df[[f]].to_numpy()
            model = make_pipeline(StandardScaler(), LogisticRegression()).fit(x[train], y[train])
            row[f] = base - log_loss(y[test], model.predict_proba(x[test])[:, 1], labels=[0, 1])
        rows.append(row)
    return pd.DataFrame(rows)


def agreement(r1: pd.Series, r2: pd.Series) -> float:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rho = spearmanr(r1.sort_index(), r2.sort_index()).statistic
    return 1.0 if np.isnan(rho) else float(rho)


def choose(rankings: dict, threshold: float = AGREEMENT, default=DEFAULT_K):
    ks = sorted(rankings)
    for i, k in enumerate(ks):
        later = ks[i + 1:]
        if later and all(agreement(rankings[k], rankings[j]) >= threshold for j in later):
            return k
    return default

### Helper checks

In [4]:
assert eligible(1000, min_fold=150) == [3, 4, 5]  # k=6 gives test folds of 142
assert eligible(581) == [3, 4]  # women's: k=5 gives test folds of 96

_r = lambda *ranks: pd.Series(ranks, index=list("abcd"), dtype=float)
assert choose({3: _r(3, 1, 2, 4), 4: _r(1, 2, 3, 4), 5: _r(1, 2, 3, 4)}) == 4  # k=3 agrees at 0.4
assert choose({3: _r(1, 2, 3, 4), 4: _r(4, 3, 2, 1), 5: _r(1, 2, 3, 4)}) == DEFAULT_K
print("Helper checks passed.")

Helper checks passed.


## Run the check

- **spread:** typical fold-to-fold SD of a feature's gain.
- **gap:** typical gap between neighbouring features in the ranking.
- **gap/spread above 1** means feature differences are bigger than fold noise.

In [5]:
df = rows.dropna(subset=CHECK_FEATURES).reset_index(drop=True)
print(f"n = {len(df)} games ({len(rows) - len(df)} dropped for missing check features)\n")

ok = eligible(len(df))
rankings, summary = {}, []
for k in K_RANGE:
    first, smallest = fold_sizes(len(df), k)
    scores = fold_scores(df, k, CHECK_FEATURES)
    means = scores.mean().sort_values(ascending=False)
    spread = scores.std(ddof=1).mean()
    gap = means.diff().abs().mean()
    if k in ok:
        rankings[k] = scores.mean().rank(ascending=False)
    summary.append({
        "k": k, "first train": first, "min test": smallest, "eligible": k in ok,
        "spread": round(spread, 4), "gap": round(gap, 4), "gap/spread": round(gap / spread, 2),
        "beat baseline": f"{(scores > 0).to_numpy().mean():.0%}",
    })

pd.DataFrame(summary).set_index("k")

n = 579 games (0 dropped for missing check features)



,first train,min test,eligible,spread,gap,gap/spread,beat baseline
k,,,,,,,
3,147,144,True,0.0327,0.0421,1.29,100%
4,119,115,True,0.0370,0.0401,1.09,100%
5,99,96,False,0.0380,0.0406,1.07,100%
6,87,82,False,0.0376,0.0404,1.07,100%
7,75,72,False,0.0385,0.0401,1.04,96%
8,67,64,False,0.0393,0.0407,1.03,100%


In [6]:
ks = sorted(rankings)
pd.DataFrame(
    [[round(agreement(rankings[a], rankings[b]), 2) for b in ks] for a in ks],
    index=pd.Index(ks, name="k"), columns=ks,
)

,3,4
k,,
3,1.0,1.0
4,1.0,1.0


In [7]:
k_rule = choose(rankings, default=None)
if k_rule is None:
    print(f"No k met the stability rule -> default k = {DEFAULT_K}")
else:
    print(f"Rule picks k = {k_rule} (smallest eligible k whose ranking matches every larger one)")

Rule picks k = 3 (smallest eligible k whose ranking matches every larger one)


## Decision: k = 4, by the rule

- **Eligible:** k = 3 and 4. At k = 5 the smallest test fold drops to 96 games, below the
  minimum of 100.
- **Stability can't tell them apart:** the ranking of the four check features is identical
  at k = 3 and 4 (agreement 1.0), and the gaps between features exceed the fold noise at both
  (gap/spread 1.26 and 1.14).
- **The tie-break, fixed in advance, takes k = 4.** That leaves 3 folds for step 5 to choose
  features on (a feature must beat the baseline in 2 of 3) and fold 4 to confirm them once.
- **Fold sizes at k = 4:** first training set 117 games, smallest test fold 116.
- **Scaled minimum gain for step 7:** 0.001 × √(957 / 581) = **0.0013**.

No score was used to choose k, and no rule was changed after the run.


In [8]:
k_rule = choose(rankings, default=None)
if k_rule is None:
    K = DEFAULT_K
    print(f"No k met the stability rule -> default k = {K}")
else:
    # Tie-break, fixed in advance: of the k values ranking exactly like the rule's pick, the largest
    same = [j for j in ks if j >= k_rule and agreement(rankings[k_rule], rankings[j]) == 1.0]
    K = max(same)
    print(f"Rule picks k = {k_rule}; identical rankings at k = {same}; tie-break -> k = {K}")
first, smallest = fold_sizes(len(df), K)
print(f"At k = {K}: first training set {first} games, smallest test fold {smallest} games.")
print(f"Step 5 chooses features on folds 1 to {K - 1} and confirms once on fold {K}.")
print(f"Scaled minimum gain for step 7: 0.001 x sqrt(957 / {len(rows)}) = {0.001 * (957 / len(rows)) ** 0.5:.4f}")


Rule picks k = 3; identical rankings at k = [3, 4]; tie-break -> k = 4
At k = 4: first training set 119 games, smallest test fold 115 games.
Step 5 chooses features on folds 1 to 3 and confirms once on fold 4.
Scaled minimum gain for step 7: 0.001 x sqrt(957 / 579) = 0.0013
